In [11]:
import pyhepmc
import os
import vector
import awkward as ak

def load_events_from_hepmc(root_file_path: str):
    hepmc_file_path  = root_file_path.replace(".root", ".hepmc").replace("reco_", "sim_")
    events = []
    if not os.path.exists(hepmc_file_path):
        print(f"Incorrect path for the .hepmc file: {hepmc_file_path}")
    with pyhepmc.open(hepmc_file_path) as f:
        for event in f:
            events.append(event)
    return events

In [6]:
path = "/home/laurits/out.hepmc"

hepmc_events = load_events_from_hepmc(path)

In [12]:
def reinitialize_p4(p4_obj: ak.Array):
    """Reinitialized the 4-momentum for particle in order to access its properties.

    Args:
        p4_obj : ak.Array
            The particle represented by its 4-momenta

    Returns:
        p4 : ak.Array
            Particle with initialized 4-momenta.
    """
    # Initialize from all the p4 fields
    name_map = {
        "x": "px",
        "y": "py",
        "z": "pz",
        "tau": "mass",
        "t": "energy",
        "rho": "pt",
    }
    p4 = vector.awk(
        ak.zip({name_map.get(field, field): p4_obj[field] for field in p4_obj.fields})
    )
    # Now make it so that the 4-vector is always saved in a similar fashion:
    p4 = vector.awk(
        ak.zip(
            {
                "pt": p4.pt,
                "eta": p4.eta,
                "phi": p4.phi,
                "energy": p4.t,
            }
        )
    )
    return p4

In [13]:
def retrieve_hepmc_gen_particles(hepmc_events):
    stable_mc_p4 = []
    stable_mc_particles = []
    for i, event in enumerate(hepmc_events):
        event_stable_gen_particles = [p for p in event.particles if (p.status == 1) and (abs(p.pid) not in [12,14,16])]
        stable_mc_particles.append([{"PDG": p.pid} for p in event_stable_gen_particles])
        stable_mc_p4.append([vector.awk(ak.zip({
            "mass": [gp.generated_mass],
            "x": [gp.momentum.px],
            "y": [gp.momentum.py],
            "z": [gp.momentum.pz]
        }))[0] for gp in event_stable_gen_particles])
    stable_mc_p4 = reinitialize_p4(ak.Array(stable_mc_p4))
    stable_mc_particles = ak.Array(stable_mc_particles)
    return stable_mc_p4, stable_mc_particles

In [25]:
hepmc_events[9]

<GenEvent momentum_unit=1, length_unit=0, event_number=9, particles=25, vertices=11, run_info=GenRunInfo(tools=[], weight_names=[], attributes={})>

In [14]:
stable_mc_p4, stable_mc_particles = retrieve_hepmc_gen_particles(hepmc_events)

In [15]:
stable_mc_particles.fields

['PDG']

In [ ]:
        for event in arrays:
            d_idx = event["_MCParticles_daughters.index"]
            d_begin = event["MCParticles.daughters_begin"]
            d_end = event["MCParticles.daughters_end"]
            tau_mask = (np.abs(event["MCParticles.PDG"]) == 15) & (
                event["MCParticles.generatorStatus"] == 2
            )
            tau_indices = np.where(tau_mask)[0]
            for tau_idx in tau_indices:
                daughter_indices = d_idx[d_begin[tau_idx] : d_end[tau_idx]]
                raw_pdgs = [int(event["MCParticles.PDG"][d]) for d in daughter_indices]
                raw_daughter_pdgs.append(raw_pdgs)

flat_pdgs = [pdg for daughters in raw_daughter_pdgs for pdg in daughters]
print(Counter(flat_pdgs).most_common(30))